# 🏗️ Bronze-Silver-Gold Data Pipeline

This notebook builds a **medallion architecture** with three layers:

| Layer | Purpose |
|-------|---------|
| **Bronze** | Raw data landing — save everything "as-is" |
| **Silver** | Clean data — fix types, remove bad rows, remove duplicates |
| **Gold** | Business-ready — aggregated reports for dashboards |

---

## What We Will Do

1. **Bronze** → Load raw sales CSV with an `ingestion_time` stamp
2. **Silver** → Clean the data (fix numbers, drop bad rows, remove duplicates)
3. **Gold** → Calculate **daily revenue by store**

In [0]:
# ============================================
# STEP 0: Setup — Read the raw CSV file
# ============================================

df_raw = spark.read.csv("/Volumes/cyntexa_dev/bronze/raw/Sales/",
                        header = True,
                        inferSchema = True
                        )
# Show first 5 rows to check what we got
print("=== Raw Data Sample ===")
df_raw.show(5 , truncate = False)

# Show the column names and guessed types
print("=== Schema (Column Types) ===")
df_raw.printSchema()

# Count total rows
print(f"Total rows loaded: {df_raw.count()}")

## 🥉 Bronze Table — Save Raw Data "As-Is"

> **Rule of Bronze:** Never change the original data. Just add metadata.

In this step we:
- Keep **all columns** exactly as they came from the source
- Add an `ingestion_timestamp` column to track when the data arrived
- Save as a **Delta table** so we can query it with SQL

In [0]:
from pyspark.sql.functions import current_timestamp

# Add a new column: when did we load this data?
df_bronze = df_raw.withColumn("ingestion_timestamp" , current_timestamp())

# Save as a Delta table (creates the table if it doesn't exist)
# "bronze.sales_raw" means: schema="bronze", table="sales_raw"
df_bronze.write\
         .mode("overwrite")\
         .saveAsTable("cyntexa_dev.bronze.sales_raw")

print("Bronze table created: bronze.sales_raw")
print(f"Bronze rows: {df_bronze.count()}")

# Show a sample
spark.table("cyntexa_dev.bronze.sales_raw").show(5 , truncate=False)